In [ ]:
from collections import Counter, defaultdict
import math

class NaiveBayesClassifier:

    def __init__(self, alpha=1.0):  # ИСПРАВЛЕНО: __init__
        """alpha — параметр сглаживания Лапласа"""
        self.alpha = alpha
        self.class_log_probs = {}
        self.word_log_probs = {}
        self.classes = []
        self.vocab = set()

    def fit(self, X, y):
        """Обучение: X — список текстов, y — список меток"""
        class_word_counts = defaultdict(lambda: Counter())
        class_total_words = Counter()
        class_counts = Counter(y)

        for text, label in zip(X, y):
            words = text.split()
            self.vocab.update(words)
            for word in words:
                class_word_counts[label][word] += 1
                class_total_words[label] += 1

        self.classes = list(class_counts.keys())
        total_docs = len(y)

        for c in self.classes:
            self.class_log_probs[c] = math.log(class_counts[c] / total_docs)

        vocab_size = len(self.vocab)
        for c in self.classes:
            self.word_log_probs[c] = {}
            total = class_total_words[c]
            for word in self.vocab:
                count = class_word_counts[c].get(word, 0)
                prob = (count + self.alpha) / (total + self.alpha * vocab_size)
                self.word_log_probs[c][word] = math.log(prob)

    def predict(self, X):
        """Предсказание меток для списка текстов"""
        predictions = []
        for text in X:
            words = text.split()
            scores = {}
            for c in self.classes:
                log_prob = self.class_log_probs[c]
                for word in words:
                    if word in self.word_log_probs[c]:
                        log_prob += self.word_log_probs[c][word]
                scores[c] = log_prob
            predictions.append(max(scores, key=scores.get))
        return predictions

    def score(self, X_test, y_test):
        """Доля правильных предсказаний"""
        preds = self.predict(X_test)
        return sum(p == t for p, t in zip(preds, y_test)) / len(y_test)

In [ ]:
import os
import string
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import Column, String, Integer

In [ ]:
# Сначала импорты и настройка (как в ваших ячейках 18-20)
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy import Column, String, Integer, create_engine

# Создание движка и сессии
engine = create_engine("sqlite:///news.db")
Session = sessionmaker(bind=engine)

# Определение модели News
Base = declarative_base()

class News(Base):
    __tablename__ = "news"

    id = Column(Integer, primary_key=True)
    title = Column(String)
    author = Column(String)
    url = Column(String)
    complexity = Column(String)
    habr_id = Column(String)
    label = Column(String)

# Теперь можно работать с БД
s = Session()
data = s.query(News).all()

print(f"len(data): {len(data)}")
print("\nПервые 3 записи:")
for news in data[:3]:
    print(f"[{news.label}, '{news.title} {news.author}']")

s.close()

In [ ]:
import string

# 1. Подключаемся к БД и берём только размеченные статьи
s = Session()
labeled_news = s.query(News).filter(News.label != None).all()
s.close()

# 2. Функция очистки текста
def clean_text(text):
    if not text:
        return ""
    # Удаляем пунктуацию и переводим в нижний регистр
    translator = str.maketrans("", "", string.punctuation)
    return text.translate(translator).lower()

# 3. Формируем признаки (X) и целевые метки (y)
X = [clean_text(f"{n.title} {n.author}") for n in labeled_news]
y = [n.label for n in labeled_news]

print(f"{len(X)} статей обработано")
print(f"{X[0]}' — '{y[0]}'")

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Обучающая выборка: {len(X_train)} статей")
print(f"Тестовая выборка: {len(X_test)} статей")

In [ ]:
model = NaiveBayesClassifier()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

In [ ]:
import csv
with open("SMSSpamCollection") as f:
        data = list(csv.reader(f, delimiter="\t"))
import string
def clean(s):
    translator = str.maketrans("", "", string.punctuation)
    return s.translate(translator)
X, y = [], []
for target, msg in data:
    X.append(msg)
    y.append(target)
X = [clean(x).lower() for x in X]
X[:3]



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
model = NaiveBayesClassifier()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

model = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', MultinomialNB(alpha=0.05)),
])

model.fit(X_train, y_train)
print(model.score(X_test, y_test))

In [ ]:
from bottle import route, template, run, request, redirect

In [ ]:
import string

# 1. Функция очистки текста
def clean(text):
    if not text: return ""
    return text.translate(str.maketrans("", "", string.punctuation)).lower()

# 2. Загружаем размеченные данные и обучаем модель
s = Session()
labeled_news = s.query(News).filter(News.label != None).all()
s.close()

X = [clean(f"{n.title} {n.author}") for n in labeled_news]
y = [n.label for n in labeled_news]

model = NaiveBayesClassifier(alpha=1.0)
model.fit(X, y)
print(f"Точность на размеченных данных: {model.score(X, y):.2%}\n")

# 3. Берем все новости из БД для ранжирования
s = Session()
all_news = s.query(News).all()
s.close()

# 4. Предсказываем метки
texts = [clean(f"{n.title} {n.author}") for n in all_news]
preds = model.predict(texts)

# 5. Объединяем новости с прогнозами и сортируем (good → maybe → never)
classified = list(zip(all_news, preds))
priority = {'good': 0, 'maybe': 1, 'never': 2}
classified.sort(key=lambda x: priority.get(x[1], 3))

# 6. Вывод в консоль
print("Ранжированный список рекомендаций")
for i, (news, pred) in enumerate(classified, 1):
    print(f"{i:2}. [{pred.upper():<5}] {news.title} | Автор: {news.author}")

#комментарий